# Ejercicio 1: tres formas de validar el mismo modelo

`viajes_features.csv` y la preparación importada de `preparacion_taxis.py`, como en las Semanas 10 y 11. Hasta ahora, cada modelo se midió una vez, sobre los mismos 595 viajes de prueba. Hoy se mide con validación cruzada, con tres esquemas distintos, y con más de una métrica. `evaluar` recibe el esquema y guarda en `resultados` el error de cada pliegue, su promedio y su desviación.

Corre todas las celdas antes de tocar nada (**Run All**): corre de principio a fin sin los `TODO` resueltos.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import GroupKFold, KFold, TimeSeriesSplit, cross_validate
from sklearn.pipeline import Pipeline
from preparacion_taxis import CATEGORICAS, NUMERICAS, OBJETIVO, YA_LISTAS, dividir, preparacion

viajes = pd.read_csv("viajes_features.csv")
COLUMNAS = NUMERICAS + YA_LISTAS + CATEGORICAS
X = viajes[COLUMNAS]
y = viajes[OBJETIVO]

# Los mismos viajes, ordenados del mas antiguo al mas reciente.
viajes_en_orden = viajes.sort_values("fecha_hora_inicio").reset_index(drop=True)
X_en_orden = viajes_en_orden[COLUMNAS]
y_en_orden = viajes_en_orden[OBJETIVO]
print("Viajes:", len(viajes), " desde", viajes_en_orden["fecha_hora_inicio"].iloc[0], "hasta", viajes_en_orden["fecha_hora_inicio"].iloc[-1])

## La función que valida cada modelo

In [ ]:
resultados = {}


def gradient_boosting():
    return GradientBoostingRegressor(random_state=42)


def bosque():
    return RandomForestRegressor(n_estimators=200, random_state=42)


def evaluar(nombre, modelo, esquema, X, y, grupos=None):
    """Validacion cruzada con el esquema dado. Guarda en resultados el error
    absoluto medio de cada pliegue, su promedio y su desviacion."""
    pipeline = Pipeline([("preparar", preparacion()), ("modelo", modelo)])
    puntajes = cross_validate(pipeline, X, y, cv=esquema, groups=grupos, scoring="neg_mean_absolute_error")
    errores = -puntajes["test_score"]
    resultados[nombre] = {
        "validacion": nombre,
        "mae_promedio": round(errores.mean(), 4),
        "desviacion": round(errores.std(), 4),
        "pliegues": [round(error, 2) for error in errores],
    }
    print(f"{nombre:28s} promedio {errores.mean():.4f}  desviacion {errores.std():.4f}  pliegues {resultados[nombre]['pliegues']}")

## Lo que dijo la Semana 10: una sola división

In [ ]:
X_ent, X_prueba, y_ent, y_prueba = dividir(viajes)
una_division = {}
for nombre, modelo in [("gradient boosting", gradient_boosting()), ("bosque aleatorio", bosque())]:
    pipeline = Pipeline([("preparar", preparacion()), ("modelo", modelo)]).fit(X_ent, y_ent)
    una_division[nombre] = mean_absolute_error(y_prueba, pipeline.predict(X_prueba))
    print(f"Una division, {nombre}: {una_division[nombre]:.4f}")

## Paso 1: validación cruzada al azar

In [ ]:
# TODO 1: validacion cruzada al azar, de 5 pliegues, mezclando los viajes
# antes de repartirlos, con random_state=42.
esquema_aleatorio = KFold(n_splits=2)

evaluar("aleatorio, gradient boosting", gradient_boosting(), esquema_aleatorio, X, y)
evaluar("aleatorio, bosque", bosque(), esquema_aleatorio, X, y)

## Paso 2: validación temporal

In [ ]:
# TODO 2: validacion temporal de 5 pliegues: cada pliegue entrena con los
# viajes anteriores y valida con los siguientes. Usa los viajes en orden.
esquema_temporal = esquema_aleatorio

evaluar("temporal, gradient boosting", gradient_boosting(), esquema_temporal, X_en_orden, y_en_orden)

## Paso 3: validación por grupo

In [ ]:
# TODO 3: validacion por grupo de 5 pliegues, con la zona de origen como
# grupo: ninguna zona aparece a la vez en entrenamiento y en validacion.
esquema_por_grupo = esquema_aleatorio
grupos = None

evaluar("por zona, gradient boosting", gradient_boosting(), esquema_por_grupo, X, y, grupos)

## Paso 4: la métrica según el costo del error

In [ ]:
modelo = Pipeline([("preparar", preparacion()), ("modelo", gradient_boosting())]).fit(X_ent, y_ent)
estimada = modelo.predict(X_prueba)
real = y_prueba.to_numpy()

mae = mean_absolute_error(real, estimada)
# TODO 4: la raiz del error cuadratico medio (rmse), y la proporcion de
# viajes de prueba en que la tarifa real supera a la estimada en mas de
# 20% (subestimados_20).
rmse = None
subestimados_20 = None

print(f"MAE {mae:.4f}   RMSE {rmse}   viajes subestimados en mas de 20%: {subestimados_20}")

## Tabla de resultados

In [ ]:
tabla = pd.DataFrame(resultados.values())
print(tabla.to_string(index=False))